In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
bn = nn.BatchNorm1d(3)

with torch.no_grad():
    bn.weight *= 5/3

print(bn.weight)
print(bn.bias)

Parameter containing:
tensor([1.6667, 1.6667, 1.6667], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)


In [12]:
linear = nn.Linear(in_features=64, out_features=128)
linear.weight.shape

torch.Size([128, 64])

In [47]:
# pytorch module with tanh activation 
class FusableLinearBlock(nn.Module):
    """A standard block: Linear -> BatchNorm -> Tanh."""
    def __init__(self,in_f,out_f):
        super().__init__()
        self.linear = nn.Linear(in_f,out_f,bias=False)
        self.bn = nn.BatchNorm1d(out_f,eps=1e-05,momentum=0.1)
        #Activation
        self.activation = nn.Tanh()
        self.is_fused = False
    def forward(self,x):
        if self.is_fused:
            return self.activation(self.linear(x))
        else :
            return self.activation(self.bn(self.linear(x)))
    def fuse(self):
        if self.is_fused:
            """Folds the BatchNorm parameters into the Linear layer."""
            return
        self.eval()
    #Parameters
        W = self.linear.weight.data
        b = self.linear.bias.data if self.linear.bias is not None else torch.zeros(W.shape[0])
        gama = self.bn.weight # gamma is refferd as this 
        beta = self.bn.bias # beta is refferd as this
        rmean = self.bn.running_mean
        rvar = self.bn.running_var
        eps = self.bn.eps
        # Sacling factor
        c = gama/torch.sqrt(rvar+eps)
        self.linear.weight.data = W * c.unsqueeze(1)
        if self.linear.bias is None:
                self.linear.bias = nn.Parameter(torch.zeros(W.shape[0]))
        self.linear.bias.data = (b - rmean)*c + beta
        self.bn = nn.Identity()
        self.is_fused = True
        print("block fused batchnorm removed")

In [48]:
# THE MULTI-LAYER PERCEPTRON (MLP) LAYER WRAPPER
class FoldableMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = FusableLinearBlock(10, 20)
        self.block2 = FusableLinearBlock(20, 20)
        self.classifier = nn.Linear(20, 2)
    def forward(self,x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)
    def fuse_net(self):
        self.block1.fuse()
        self.block2.fuse()

In [49]:
# 1. Initialize the model
model = FoldableMLP()

# 2. Set to evaluation mode (CRITICAL for BatchNorm to use running stats instead of batch stats)
model.eval()

# 3. Create a dummy input tensor (Batch Size of 4, Input Features of 10)
dummy_input = torch.randn(4, 10)

# 4. Get the output BEFORE fusion
with torch.no_grad():
    output_before = model(dummy_input)

# 5. Fuse the network
model.fuse_net()

# 6. Get the output AFTER fusion
with torch.no_grad():
    output_after = model(dummy_input)

# 7. Compare the results
# torch.allclose checks if all elements are equal within a small tolerance (due to floating point math)
is_identical = torch.allclose(output_before, output_after, atol=1e-5)

print("\n--- Test Results ---")
print(f"Output Before Fusion:\n{output_before}\n")
print(f"Output After Fusion:\n{output_after}\n")
print(f"Are the outputs mathematically identical? {'✅ YES' if is_identical else '❌ NO'}")

block fused batchnorm removed
block fused batchnorm removed

--- Test Results ---
Output Before Fusion:
tensor([[-0.0679, -0.0923],
        [-0.0318, -0.0928],
        [-0.0625, -0.1305],
        [ 0.0084, -0.7272]])

Output After Fusion:
tensor([[-0.0679, -0.0923],
        [-0.0318, -0.0928],
        [-0.0625, -0.1305],
        [ 0.0084, -0.7272]])

Are the outputs mathematically identical? ✅ YES
